# 第 1 周结束练习 —— 技术问答助手

## 练习目标（理念）

为展示你对 **OpenAI API** 与本地 **Ollama** 的熟悉程度，做一个小工具：

- **输入**：一个技术问题（例如「这段 Python 代码在干什么？」）
- **输出**：清晰、简洁的 Markdown 解释
- **额外要求**：用**流式（streaming）**一边生成一边更新显示，而不是等整段答完才一次性打印

这是你在课程期间自己也能天天用的助手：遇到看不懂的代码，丢进来问模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块拼接并用 `update_display` 刷新 |
| OpenAI 云端模型 | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），经 OpenAI 兼容 `/v1` 接口 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；若要用 Llama，需本机 Ollama 在 `http://localhost:11434` 运行，并已 `ollama pull llama3.2`
3. 在「提问」单元格改写 `question`，再分别跑 GPT 与 Llama 两格，对比回答风格


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：本练习主体未直接用到，保留原导入以免改逻辑
import json
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、初次 display、流式 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：同一套 SDK 可打云端，也可打本地 Ollama 兼容端点
from openai import OpenAI


In [ ]:
# ========== 常量：模型名字与 Ollama 地址集中写在一处 ==========

# OpenAI 云端小模型：便宜、够用，适合做解释类问答
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；字符串必须和本机已安装的模型名一致
MODEL_LLAMA = 'llama3.2'

# Ollama 的 OpenAI 兼容基址（注意是 /v1，不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [ ]:
# ========== 环境 + 双客户端：云端 OpenAI 与本地 Ollama（兼容 SDK）==========

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 读取常见的 OPENAI_API_KEY（注意名字，不是 OPENAI_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：存在、以 sk-proj- 开头、长度够 —— 只给提示，不抛异常
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 默认 OpenAI 客户端：密钥从环境变量自动读取
openai = OpenAI()

# Ollama 也用 OpenAI()，但 base_url 指向本地；api_key 占位字符串 "ollama"（本地通常不校验）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


In [ ]:
# ========== 系统提示：定「技术助手」人设与输出格式（发给模型的英文勿译）==========

helper_system_prompt = """
You are an efficient technical assistant that analyzes the user's questions related to AI programming 
and answers them thourougly but optimizing tokens output to keep answers concise. 
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
""" 


In [ ]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
# 练习建议：换成你自己今天看不懂的一行代码，再分别跑下面 GPT / Llama 两格做对比
question = """
Please explain what this code does and why: 
yield from {book.get("author") for book in books if book.get("author")}
"""


In [ ]:
# ========== 流式回答：同一函数可喂 openai 或 ollama 客户端 ==========

def stream_answer(client, model, question):
    # stream=True：服务端持续推送增量 delta，而不是等整段生成完
    stream = client.chat.completions.create(
        model= model,
        messages=[
            # system：助手人设与格式约束
            {"role": "system", "content": helper_system_prompt},
            # user：真正的用户问题
            {"role": "user", "content": question}
          ],
        stream=True
    )    
    # response：累积已收到的全部文本，用于每次刷新整段 Markdown
    response = ""
    # display_id=True：拿到可更新的显示句柄，后续用 update_display 原地刷新
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # delta.content 可能是 None（例如结束块）；用 or '' 避免拼接报错
        response += chunk.choices[0].delta.content or ''
        # 每来一块就用完整累积文本刷新同一显示区域（打字机效果）
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini 流式回答 ==========

stream_answer(openai, MODEL_GPT, question)


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（经 Ollama 兼容接口）流式回答 ==========

stream_answer(ollama, MODEL_LLAMA, question)
